In [1]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 17.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 78.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 64.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 1.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 22.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 11.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 47.2 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: 

In [3]:
import os
import json
from PIL import Image
import matplotlib.pyplot as plt
import torch
from torch import nn,optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchinfo import summary
from scipy.io import loadmat
import numpy as np
import tqdm
import matplotlib.pyplot as plt
import numpy as np
import torch.backends.cudnn as cudnn
import cv2
import random
import torch.nn.functional as F
import torchvision
import cv2
from ultralytics import YOLO

class AverageMeter(object):
    """ Computes and stores the average and current value.
    Can handle both scalar values and numpy arrays.
    """
    def __init__(self, unit='-', is_vector=False):
        self.reset()
        self.unit = unit
        self.is_vector = is_vector

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0
        
    def update(self, val, n=1):
        self.val = val
        
        # Handle initialization for vectors
        if self.count == 0 and isinstance(val, np.ndarray):
            self.sum = np.zeros_like(val, dtype=np.float64)
            
        # Update sum and compute average
        if isinstance(val, np.ndarray):
            self.sum = self.sum + val * n
        else:
            self.sum += val * n
            
        self.count += n
        
        if self.count > 0:
            if isinstance(self.sum, np.ndarray):
                self.avg = self.sum / self.count
            else:
                self.avg = self.sum / self.count
                
    def magnitude(self):
        """Return magnitude for vector quantities"""
        if isinstance(self.avg, np.ndarray):
            return np.linalg.norm(self.avg)
        return self.avg


def dcm2quat(dcm):
        """ Computing quaternion from direction cosine matrix, inverse of quat2dcm.
        Arguments:
            dcm: (3,3) numpy.ndarray - direction cosine matrix
        Returns:
            q: (4,) numpy.ndarray - unit quaternion (scalar-first) [w, x, y, z]
        """
        # Handle batch input
        if dcm.ndim == 3:
            return np.array([dcm2quat(dcm_i) for dcm_i in dcm])
        
        # Compute trace of the matrix
        trace = np.trace(dcm)
        
        if trace > 0:
            # Trace is positive
            S = np.sqrt(trace + 1.0) * 2
            q0 = 0.25 * S
            q1 = (dcm[2, 1] - dcm[1, 2]) / S
            q2 = (dcm[0, 2] - dcm[2, 0]) / S
            q3 = (dcm[1, 0] - dcm[0, 1]) / S
        elif dcm[0, 0] > dcm[1, 1] and dcm[0, 0] > dcm[2, 2]:
            S = np.sqrt(1.0 + dcm[0, 0] - dcm[1, 1] - dcm[2, 2]) * 2
            q0 = (dcm[2, 1] - dcm[1, 2]) / S
            q1 = 0.25 * S
            q2 = (dcm[0, 1] + dcm[1, 0]) / S
            q3 = (dcm[0, 2] + dcm[2, 0]) / S
        elif dcm[1, 1] > dcm[2, 2]:
            S = np.sqrt(1.0 + dcm[1, 1] - dcm[0, 0] - dcm[2, 2]) * 2
            q0 = (dcm[0, 2] - dcm[2, 0]) / S
            q1 = (dcm[0, 1] + dcm[1, 0]) / S
            q2 = 0.25 * S
            q3 = (dcm[1, 2] + dcm[2, 1]) / S
        else:
            S = np.sqrt(1.0 + dcm[2, 2] - dcm[0, 0] - dcm[1, 1]) * 2
            q0 = (dcm[1, 0] - dcm[0, 1]) / S
            q1 = (dcm[0, 2] + dcm[2, 0]) / S
            q2 = (dcm[1, 2] + dcm[2, 1]) / S
            q3 = 0.25 * S
        
        # Form quaternion and normalize
        q = np.array([q0, q1, q2, q3])
        q = q / np.linalg.norm(q)
        
        return q

def pnp(points_3d, points_2d, camera_matrix, dist_coeffs, method=cv2.SOLVEPNP_ITERATIVE):

    assert points_3d.shape[0] == points_2d.shape[0], 'points 3D and points 2D must have same number of vertices'
    if method==cv2.SOLVEPNP_EPNP:
        points_3d=np.expand_dims(points_3d, 0)
        points_2d=np.expand_dims(points_2d, 0)

    points_2d = np.ascontiguousarray(points_2d.astype(np.float64))
    points_3d = np.ascontiguousarray(points_3d.astype(np.float64))
    camera_matrix = camera_matrix.astype(np.float64)
    _, R_exp, t = cv2.solvePnP(points_3d,
                               points_2d,
                               camera_matrix,
                               dist_coeffs,
                               flags=method)
    R, _ = cv2.Rodrigues(R_exp)
    
    return np.concatenate([R, t], axis=-1)

def error_translation(t_pr, t_gt):
    t_pr = np.reshape(t_pr, (3,))
    t_gt = np.reshape(t_gt, (3,))

    return t_gt - t_pr

def error_orientation(q_pr, q_gt):
    # q must be [qvec, qcos]
    q_pr = np.reshape(q_pr, (4,))
    q_gt = np.reshape(q_gt, (4,))

    qdot = np.abs(np.dot(q_pr, q_gt))
    qdot = np.minimum(qdot, 1.0)
    return np.rad2deg(2*np.arccos(qdot)) # [deg]

def speed_score(t_pr, q_pr, t_gt, q_gt, applyThresh=True, rotThresh=0.5, posThresh=0.005):
    # rotThresh: rotation threshold [deg]
    # posThresh: normalized translation threshold [m/m]
    err_t = error_translation(t_pr, t_gt)
    err_t = np.linalg.norm(err_t)
    err_q = error_orientation(q_pr, q_gt) # [deg]

    t_gt = np.reshape(t_gt, (3,))
    speed_t = err_t / np.sqrt(np.sum(np.square(t_gt)))
    speed_q = np.deg2rad(err_q)

    # Check if within threshold
    if applyThresh and err_q < rotThresh:
        speed_q = 0.0

    if applyThresh and speed_t < posThresh:
        speed_t = 0.0

    speed = speed_t + speed_q

    # Accuracy of within threshold
    acc   = float(err_q < rotThresh and speed_t < posThresh)

    return speed, acc


class SPEEDDataset(Dataset):
    def __init__(self, images_dir, json_dir, tango_points_dir, camera_matrix_dir, transform=None, is_train=True):
        self.image_width = 1920
        self.image_height = 1200
        self.aspect_ratio = self.image_width * 1.0 / self.image_height
        self.pixel_std = 200
        self.num_joints = 11
        self.image_size = [224,224]
        self.scale_factor = 0.25
        self.rotation_factor = 30
        self.sigma = 2
        self.is_train = is_train
        self.transform = transform

        self.imagesList = []
        self.frameInfoList = []
        self.images_dir = images_dir
        self.json_dir = json_dir

        self.keypts3d = self.load_tango_3d_keypoints(tango_points_dir) # https://www.desmos.com/3d/jud6lng9gn
        self.cameraMatrix, self.distCoeffs = self.load_camera_intrinsics(camera_matrix_dir)

        with open(self.json_dir, 'r') as f:
            annotations = json.load(f)
            lookup = { item['filename']: item for item in annotations }
            cnt = 0
            for filename in tqdm.tqdm(sorted(os.listdir(self.images_dir))):
                if filename not in lookup:
                    continue
                frame_idx = int(''.join(filter(str.isdigit, filename.split('.')[0])))
                self.imagesList.append(os.path.join(self.images_dir, filename))

                q_vbs2tango = np.array(lookup[filename]["q_vbs2tango"], dtype=np.float32)
                r_Vo2To_vbs = np.array(lookup[filename]['r_Vo2To_vbs_true'], dtype=np.float32)

                frame_info = {
                    'filename': filename,
                    'q_vbs2tango': q_vbs2tango,
                    'r_Vo2To_vbs': r_Vo2To_vbs
                }
                self.frameInfoList.append(frame_info)

                cnt = cnt + 1
                if cnt>100:
                    break

    def __len__(self):
        return len(self.imagesList)
    
    def quat2dcm(self, q):
        """ Computing direction cosine matrix from quaternion, adapted from PyNav.
        Arguments:
            q: (4,) numpy.ndarray - unit quaternion (scalar-first)
        Returns:
            dcm: (3,3) numpy.ndarray - corresponding DCM
        """

        # normalizing quaternion
        q = q / np.linalg.norm(q)

        q0, q1, q2, q3 = q[0], q[1], q[2], q[3]
        dcm = np.zeros((3, 3))
        dcm[0, 0] = 2 * q0 ** 2 - 1 + 2 * q1 ** 2
        dcm[1, 1] = 2 * q0 ** 2 - 1 + 2 * q2 ** 2
        dcm[2, 2] = 2 * q0 ** 2 - 1 + 2 * q3 ** 2
        dcm[0, 1] = 2 * q1 * q2 + 2 * q0 * q3
        dcm[0, 2] = 2 * q1 * q3 - 2 * q0 * q2
        dcm[1, 0] = 2 * q1 * q2 - 2 * q0 * q3
        dcm[1, 2] = 2 * q2 * q3 + 2 * q0 * q1
        dcm[2, 0] = 2 * q1 * q3 + 2 * q0 * q2
        dcm[2, 1] = 2 * q2 * q3 - 2 * q0 * q1

        return dcm

    def load_tango_3d_keypoints(self, mat_dir):
        vertices = loadmat(mat_dir)['tango3Dpoints']
        corners3D = np.transpose(np.array(vertices, dtype=np.float32))
        return corners3D

    def load_camera_intrinsics(self, camera_json):
        with open(camera_json) as f:
            cam = json.load(f)
        cameraMatrix = np.array(cam['cameraMatrix'], dtype=np.float32)
        distCoeffs = np.array(cam['distCoeffs'], dtype=np.float32)
        return cameraMatrix, distCoeffs

    def project_keypoints(self, q_vbs2tango, r_Vo2To_vbs, cameraMatrix, distCoeffs, keypoints):
        if keypoints.shape[0] != 3:
            keypoints = np.transpose(keypoints)
        keypoints = np.vstack((keypoints, np.ones((1, keypoints.shape[1]))))
        pose_mat = np.hstack((np.transpose(self.quat2dcm(q_vbs2tango)),
                              np.expand_dims(r_Vo2To_vbs, 1)))
        xyz = np.dot(pose_mat, keypoints)
        x0, y0 = xyz[0, :] / xyz[2, :], xyz[1, :] / xyz[2, :]
        r2 = x0 * x0 + y0 * y0
        cdist = 1 + distCoeffs[0] * r2 + distCoeffs[1] * r2 * r2 + distCoeffs[4] * r2 * r2 * r2
        x = x0 * cdist + distCoeffs[2] * 2 * x0 * y0 + distCoeffs[3] * (r2 + 2 * x0 * x0)
        y = y0 * cdist + distCoeffs[2] * (r2 + 2 * y0 * y0) + distCoeffs[3] * 2 * x0 * y0
        points2D = np.vstack((cameraMatrix[0, 0] * x + cameraMatrix[0, 2],
                              cameraMatrix[1, 1] * y + cameraMatrix[1, 2]))
        return points2D

    def _box2cs(self, box):
        x, y, w, h = box[:4]
        return self._xywh2cs(x, y, w, h)

    def _xywh2cs(self, x, y, w, h):
        center = np.zeros((2), dtype=np.float32)
        center[0] = x + w * 0.5
        center[1] = y + h * 0.5

        if w > self.aspect_ratio * h:
            h = w * 1.0 / self.aspect_ratio
        elif w < self.aspect_ratio * h:
            w = h * self.aspect_ratio
        scale = np.array(
            [w * 1.0 / self.pixel_std, h * 1.0 / self.pixel_std],
            dtype=np.float32)
        if center[0] != -1:
            scale = scale * 1.25

        return center, scale

    def get_affine_transform(self, center, scale, rot, output_size, shift=np.array([0, 0], dtype=np.float32), inv=0):
        if not isinstance(scale, np.ndarray) and not isinstance(scale, list):
            scale = np.array([scale, scale])
        scale_tmp = scale * 200.0
        src_w = scale_tmp[0]
        dst_w = output_size[0]
        dst_h = output_size[1]
        rot_rad = np.pi * rot / 180
        src_dir = self.get_dir([0, src_w * -0.5], rot_rad)
        dst_dir = np.array([0, dst_w * -0.5], np.float32)
        src = np.zeros((3, 2), dtype=np.float32)
        dst = np.zeros((3, 2), dtype=np.float32)
        src[0, :] = center + scale_tmp * shift
        src[1, :] = center + src_dir + scale_tmp * shift
        dst[0, :] = [dst_w * 0.5, dst_h * 0.5]
        dst[1, :] = np.array([dst_w * 0.5, dst_h * 0.5]) + dst_dir
        src[2:, :] = self.get_3rd_point(src[0, :], src[1, :])
        dst[2:, :] = self.get_3rd_point(dst[0, :], dst[1, :])
        if inv:
            trans = cv2.getAffineTransform(np.float32(dst), np.float32(src))
        else:
            trans = cv2.getAffineTransform(np.float32(src), np.float32(dst))
        return trans

    def affine_transform(self, pt, t):
        new_pt = np.array([pt[0], pt[1], 1.]).T
        new_pt = np.dot(t, new_pt)
        return new_pt[:2]
    
    def get_3rd_point(self, a, b):
        direct = a - b
        return b + np.array([-direct[1], direct[0]], dtype=np.float32)
    
    def get_dir(self, src_point, rot_rad):
        sn, cs = np.sin(rot_rad), np.cos(rot_rad)
        src_result = [0, 0]
        src_result[0] = src_point[0] * cs - src_point[1] * sn
        src_result[1] = src_point[0] * sn + src_point[1] * cs
        return src_result

    def __getitem__(self, idx):
        image_path = self.imagesList[idx]
        data_numpy = cv2.imread(image_path, cv2.IMREAD_COLOR | cv2.IMREAD_IGNORE_ORIENTATION)
        data_numpy = cv2.cvtColor(data_numpy, cv2.COLOR_BGR2RGB)

        q_vbs2tango = self.frameInfoList[idx]['q_vbs2tango']
        r_Vo2To_vbs = self.frameInfoList[idx]['r_Vo2To_vbs']

        keypts2d = self.project_keypoints(q_vbs2tango, r_Vo2To_vbs,
                                          self.cameraMatrix, self.distCoeffs,
                                          self.keypts3d)  # (2, 11)

        xmin = float(np.min(keypts2d[0]))
        xmax = float(np.max(keypts2d[0]))
        ymin = float(np.min(keypts2d[1]))
        ymax = float(np.max(keypts2d[1]))

        # clamp bbox to image boundaries
        xmin = max(0.0, xmin)
        ymin = max(0.0, ymin)
        xmax = min(self.image_width, xmax)
        ymax = min(self.image_height, ymax)

        box_full = [0, 0, self.image_width, self.image_height]

        c_full, s_full = self._box2cs(box_full)
        r_full = 0

        if self.is_train:
            sf = self.scale_factor
            rf = self.rotation_factor
            s = s * np.clip(np.random.randn()*sf + 1, 1 - sf, 1 + sf)
            r = np.clip(np.random.randn()*rf, -rf*2, rf*2) \
                    if random.random() <= 0.6 else 0
            s_full = s_full * np.clip(np.random.randn()*sf + 1, 1 - sf, 1 + sf)
            r_full = np.clip(np.random.randn()*rf, -rf*2, rf*2) \
                    if random.random() <= 0.6 else 0
            
        trans_full = self.get_affine_transform(c_full, s_full, r_full, self.image_size)
        
        wrappedImage_full = cv2.warpAffine(
            data_numpy,
            trans_full,
            (int(self.image_size[0]), int(self.image_size[1])),
            flags=cv2.INTER_LINEAR)

        keypts = np.zeros_like(keypts2d)
        for i in range(keypts2d.shape[1]):
            keypts[:, i] = self.affine_transform(keypts2d[:, i], trans_full)

        if self.transform:
            wrappedImage_full = self.transform(wrappedImage_full)
        keypts = torch.tensor(keypts, dtype=torch.float32)
        return wrappedImage_full, keypts, q_vbs2tango, r_Vo2To_vbs, trans_full

val_transforms = transforms.Compose([
    transforms.ToTensor(),
    # transforms.Normalize(
    #     mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
    # )
])

syntheticdataset = SPEEDDataset(images_dir='/kaggle/input/speedsplit/speed/images/trainval', json_dir='/kaggle/input/speedsplit/speed/val.json', tango_points_dir='/kaggle/input/mat-file/tangoPoints.mat', camera_matrix_dir='/kaggle/input/mat-file/camera.json', transform=val_transforms, is_train = False)
realdataset = SPEEDDataset(images_dir='/kaggle/input/speedsplit/speed/images/real', json_dir='/kaggle/input/speedsplit/speed/real.json', tango_points_dir='/kaggle/input/mat-file/tangoPoints.mat', camera_matrix_dir='/kaggle/input/mat-file/camera.json', transform=val_transforms, is_train = False)

class ApproachZero(nn.Module):
    def __init__(self):
        super().__init__()

        self.num_joints = 11

        self.resnet50 = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V2)
        self.resnet50Features = nn.Sequential(*list(self.resnet50.children())[:-2])
        # Running features [B, 2048, 7, 7]
        
        self.conv1 = nn.Conv2d(2048, 512, kernel_size=1)
        self.bn1   = nn.BatchNorm2d(512)
        self.relu  = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(512, 2 * self.num_joints, kernel_size=1)
        
        self.conv3 = nn.Conv2d(2 * self.num_joints, 2 * self.num_joints, kernel_size=3, stride=2, padding=1)  
        self.bn3 = nn.BatchNorm2d(2 * self.num_joints)
        
        self.conv4 = nn.Conv2d(2 * self.num_joints, 2 * self.num_joints, kernel_size=3, stride=2, padding=1)  
        self.bn4 = nn.BatchNorm2d(2 * self.num_joints)
        
        self.conv5 = nn.Conv2d(2 * self.num_joints, 2 * self.num_joints, kernel_size=2, stride=2)  

    def forward(self, X, targets=None):
        feats = self.resnet50Features(X)
        x = self.conv1(feats)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.conv2(x)
        
        x = self.conv3(x)
        x = self.bn3(x)
        x = self.relu(x)
        
        x = self.conv4(x)
        x = self.bn4(x)
        x = self.relu(x)
        
        x = self.conv5(x)
        x = self.relu(x)
        
        return x
   
def visualize_keypoints(image, pred_keypoints, gt_keypoints=None, save_path=None, idx=0):
    """
    Visualize keypoints on an image
    Args:
        image: PIL Image or numpy array
        pred_keypoints: predicted keypoints (n_points, 2)
        gt_keypoints: ground truth keypoints (n_points, 2), optional
        save_path: path to save the visualization
        idx: sample index for filename
    """
    plt.figure(figsize=(12, 8))
    
    # Convert tensor to numpy for plotting
    if isinstance(image, torch.Tensor):
        img_np = image.permute(1, 2, 0).cpu().numpy()
        # Rescale if normalized
        if img_np.max() <= 1:
            img_np = (img_np * 255).astype(np.uint8)
    else:
        img_np = image
        
    plt.imshow(img_np)
    
    # Plot predicted keypoints in red
    if pred_keypoints is not None:
        plt.scatter(pred_keypoints[:, 0], pred_keypoints[:, 1], c='r', marker='x', s=40, label='Predicted')
    
    # Plot ground truth keypoints in green
    if gt_keypoints is not None:
        plt.scatter(gt_keypoints[:, 0], gt_keypoints[:, 1], c='g', marker='o', s=40, alpha=0.5, label='Ground Truth')
    
    plt.legend()
    plt.title(f"Keypoints Visualization - Sample {idx}")
    
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path)
        plt.close()
    else:
        plt.show()

def test_loop(test_dataloader, modelODN, model, device, camera_matrix, dist_coeffs, points_3d, save_viz=True, viz_dir="keypoint_viz"):
    # Initialize error meters and lists for median
    err_q_meter     = AverageMeter('deg')
    err_t_meter     = AverageMeter('m',is_vector=True)
    speed_score_meter = AverageMeter('-')
    speed_acc_meter = AverageMeter('-')
    q_errors_all = []
    t_errors_mag_all = []
    
    # Create visualization directory if needed
    if save_viz:
        os.makedirs(viz_dir, exist_ok=True)

    model.eval()
    
    for batch, (X, Y_keypts, q_gt, t_gt, trans_full) in enumerate(test_dataloader):
        X, Y_keypts, q_gt, t_gt = X.to(device), Y_keypts.to(device), q_gt.to(device), t_gt.to(device)
        B = X.shape[0]

        YOLO_output = modelODN(X, verbose=False)
        
        if YOLO_output[0].boxes.xyxy.shape[0] > 0:
            box = YOLO_output[0].boxes.xyxy[0].cpu()
        else:
            _, _, h, w = X.shape
            box = torch.tensor([0., 0., w, h])
        
        x1, y1, x2, y2 = box.int()
        
        x1 = max(0, x1)
        y1 = max(0, y1)
        x2 = min(X.shape[3], x2)
        y2 = min(X.shape[2], y2)
        
        crop = X[0, :, y1:y2, x1:x2]
        
        crop = F.interpolate(crop.unsqueeze(0), size=(224, 224), mode='bilinear', align_corners=False)
        
        with torch.no_grad():
            pred = model(crop)
            pred_keypts = pred.reshape(1, 2, model.num_joints)
        
        scale_x = (x2 - x1) / 224.0
        scale_y = (y2 - y1) / 224.0
        
        pred_keypts[0, 0, :] = pred_keypts[0, 0, :] * scale_x + x1
        pred_keypts[0, 1, :] = pred_keypts[0, 1, :] * scale_y + y1

        pred_keypts_np = pred_keypts.cpu().numpy()
        pred_keypts_np = pred_keypts_np[0]
        pred_keypts_np = pred_keypts_np.transpose(1, 0)  

        inv_trans = cv2.invertAffineTransform(trans_full[0].cpu().numpy())
        
        # Transform keypoints back to original image space
        keypts = np.zeros((pred_keypts_np.shape[0], 2))
        for i in range(pred_keypts_np.shape[0]):
            keypts[i] = test_dataloader.dataset.affine_transform(pred_keypts_np[i], inv_trans)
            
        # Get ground truth keypoints in original image space
        gt_keypts = Y_keypts[0].cpu().numpy().transpose(1, 0)  # (11, 2)
        
        # Visualize some samples
        if save_viz and (batch < 10 or batch % 50 == 0):  # Save first 10 and then every 50th sample
            orig_img = X[0].cpu()  # Get the original image tensor
            
            # Create visualization with both sets of keypoints
            viz_filename = f"{viz_dir}/sample_{batch}.png"
            visualize_keypoints(
                orig_img, 
                pred_keypts_np,  # These are in transformed image space
                gt_keypts,       # Ground truth keypoints
                save_path=viz_filename, 
                idx=batch
            )
            
            # Also visualize keypoints in original image space
            if batch < 5:  # Only for a few samples
                # Get original image by inverse transform
                # This is a simplified approach - for accuracy, you'd need to apply the actual inverse transform to the image
                orig_viz_filename = f"{viz_dir}/orig_sample_{batch}.png"
                # This would require getting the original image from the dataset
                orig_path = test_dataloader.dataset.imagesList[batch]
                orig_img = cv2.imread(orig_path)
                orig_img = cv2.cvtColor(orig_img, cv2.COLOR_BGR2RGB)
                visualize_keypoints(
                    orig_img,
                    keypts,          # Predicted keypoints in original space
                    None,            # We don't have GT in original space easily accessible
                    save_path=orig_viz_filename,
                    idx=batch
                )

        RT = pnp(points_3d, keypts, camera_matrix, dist_coeffs)

        R = RT[:, :3]
        t = RT[:, 3:]

        q_pr = dcm2quat(R)  
        t_pr = t

        # Ground-truth
        q_gt_i = q_gt[0].cpu().numpy()
        t_gt_i = t_gt[0].cpu().numpy()

        # Metrics
        err_q = error_orientation(q_pr, q_gt_i) # [deg]
        err_t = error_translation(t_pr, t_gt_i)

        speed, acc = speed_score(t_pr, q_pr, t_gt_i, q_gt_i,applyThresh=True, rotThresh=0.169, posThresh=0.002173)

        # Collect errors for median
        q_errors_all.append(err_q)
        t_errors_mag_all.append(np.linalg.norm(err_t))
        speed_score_meter.update(speed, B)
        speed_acc_meter.update(acc, B)

        err_q_meter.update(err_q, B)
        err_t_meter.update(err_t, B)

        print(f"\rBatch {batch+1}/{len(test_dataloader)}: Orientation Error: {err_q_meter.val:.2f} {err_q_meter.unit}, Translation Error: [{err_t[0]:.2f}, {err_t[1]:.2f}, {err_t[2]:.2f}] (mag: {np.linalg.norm(err_t_meter.val):.2f}) {err_t_meter.unit}                                                 ", end="", flush=True)

    performances = {
        'eR': err_q_meter,
        'eT': err_t_meter,
        'speed': speed_score_meter,
        'acc': speed_acc_meter
    }
    # Compute median metrics
    medians = {
        'eR_med': np.median(q_errors_all),
        'eT_med': np.median(t_errors_mag_all)
    }
    return performances, medians

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
modelODN = YOLO("/kaggle/input/yolospeed/runs/detect/train/weights/best.pt").to(device)
model = ApproachZero().to(device)
checkpoint = torch.load("/kaggle/input/yoloextend/checkpoint.pth", weights_only=False, map_location=torch.device('cpu') ) 
model.load_state_dict(checkpoint['state_dict'])


batch_size = 1

test_syn_dataloader = DataLoader(syntheticdataset, batch_size, shuffle=False, num_workers=1, pin_memory=True, drop_last=True)
test_real_dataloader = DataLoader(realdataset, batch_size, shuffle=False, num_workers=1, pin_memory=True, drop_last=True)

print("Testing on synthetic dataset...")
performances_syn, medians_syn = test_loop(test_syn_dataloader, modelODN, model, device, 
                                         syntheticdataset.cameraMatrix, syntheticdataset.distCoeffs, 
                                         syntheticdataset.keypts3d, save_viz=True, viz_dir="keypoint_viz_synthetic")
print("\n")
print("\nTesting on real dataset...")
performances_real, medians_real = test_loop(test_real_dataloader, modelODN, model, device, 
                                           realdataset.cameraMatrix, realdataset.distCoeffs, 
                                           realdataset.keypts3d, save_viz=True, viz_dir="keypoint_viz_real")

print("\n")
print("\nResults Summary:")
print("+" + "-"*22 + "+" + "-"*30 + "+" + "-"*30 + "+")
print(f"| {'Metric':<20} | {'SPEED synthetic test-set':<28} | {'SPEED real test-set':<28} |")
print("+" + "-"*22 + "+" + "-"*30 + "+" + "-"*30 + "+")

mean_et_syn = f"[{performances_syn['eT'].avg[0]:.3f} {performances_syn['eT'].avg[1]:.3f} {performances_syn['eT'].avg[2]:.3f}]"
mean_et_real = f"[{performances_real['eT'].avg[0]:.3f} {performances_real['eT'].avg[1]:.3f} {performances_real['eT'].avg[2]:.3f}]"
print(f"| {'Mean ET (m)':<20} | {mean_et_syn:<28} | {mean_et_real:<28} |")


et_mag_syn = np.linalg.norm(performances_syn['eT'].avg)
et_mag_real = np.linalg.norm(performances_real['eT'].avg)
print(f"| {'Mean ET mag (m)':<20} | {et_mag_syn:<28.4f} | {et_mag_real:<28.4f} |")


median_et_syn = f"{medians_syn['eT_med']:.3f}"
median_et_real = f"{medians_real['eT_med']:.3f}"
print(f"| {'Median ET mag (m)':<20} | {median_et_syn:<28} | {median_et_real:<28} |")


print(f"| {'Mean ER (deg)':<20} | {performances_syn['eR'].avg:<28.4f} | {performances_real['eR'].avg:<28.4f} |")
print(f"| {'Median ER (deg)':<20} | {medians_syn['eR_med']:<28.4f} | {medians_real['eR_med']:<28.4f} |")
print("+" + "-"*22 + "+" + "-"*30 + "+" + "-"*30 + "+")
print(f"| {'Speed Score':<20} | {performances_syn['speed'].avg:<28.4f} | {performances_real['speed'].avg:<28.4f} |")
print(f"| {'Accuracy':<20} | {performances_syn['acc'].avg:<28.4f} | {performances_real['acc'].avg:<28.4f} |")
print("+" + "-"*22 + "+" + "-"*30 + "+" + "-"*30 + "+")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


100%|██████████| 5/5 [00:00<00:00, 25206.15it/s]
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 171MB/s]


Testing on synthetic dataset...
Batch 101/101: Orientation Error: 36.85 deg, Translation Error: [0.17, -0.17, -10.51] (mag: 10.51) m                                                 


Testing on real dataset...
Batch 5/5: Orientation Error: 69.82 deg, Translation Error: [0.20, 0.07, -2.46] (mag: 2.47) m                                                     


Results Summary:
+----------------------+------------------------------+------------------------------+
| Metric               | SPEED synthetic test-set     | SPEED real test-set          |
+----------------------+------------------------------+------------------------------+
| Mean ET (m)          | [-0.006 0.093 6.007]         | [-0.282 0.035 4.846]         |
| Mean ET mag (m)      | 6.0073                       | 4.8544                       |
| Median ET mag (m)    | 0.868                        | 7.034                        |
| Mean ER (deg)        | 43.8142                      | 130.8098                     |
| Median ER (d